In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor, VotingRegressor
from sklearn.linear_model import LinearRegression

In [12]:
data = fetch_california_housing()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [13]:
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("Decision Tree RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_dt)))
print("Decision Tree R2:", r2_score(y_test, y_pred_dt))


Decision Tree RMSE: 0.7037294974840077
Decision Tree R2: 0.622075845135081


# Bagging Regressor

In [14]:
bag = BaggingRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=50,
    random_state=42
)

bag.fit(X_train, y_train)
y_pred_bag = bag.predict(X_test)

print("Bagging RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_bag)))

Bagging RMSE: 0.5072463267331407


rnadom forest

In [15]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))

Random Forest RMSE: 0.544511508935255


Gradient Boosting

In [16]:
gb = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print("Gradient Boosting RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_gb)))

Gradient Boosting RMSE: 0.5422152016168362


# Voting Regressor

In [8]:
vote = VotingRegressor(
    estimators=[
        ('lr', LinearRegression()),
        ('rf', RandomForestRegressor()),
        ('gb', GradientBoostingRegressor())
    ]
)

vote.fit(X_train, y_train)
y_pred_vote = vote.predict(X_test)

print("Voting Regressor RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_vote)))

Voting Regressor RMSE: 0.5549100072701251


In [9]:
models = ["Decision Tree", "Bagging", "Random Forest", "Gradient Boosting", "Voting"]

rmse_scores = [
    np.sqrt(mean_squared_error(y_test, y_pred_dt)),
    np.sqrt(mean_squared_error(y_test, y_pred_bag)),
    np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    np.sqrt(mean_squared_error(y_test, y_pred_gb)),
    np.sqrt(mean_squared_error(y_test, y_pred_vote))
]

df = pd.DataFrame({"Model": models, "RMSE": rmse_scores})
print(df)


               Model      RMSE
0      Decision Tree  0.703729
1            Bagging  0.507246
2      Random Forest  0.544512
3  Gradient Boosting  0.542215
4             Voting  0.554910


# Hyperparameter Tuning (Random Forest)

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(RandomForestRegressor(), param_grid, cv=3,
                    scoring='neg_mean_squared_error')

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

best_rf = grid.best_estimator_
y_pred_best = best_rf.predict(X_test)

print("Tuned RF RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_best)))